In [1]:
'''
task: classify syllogism validity with TFLPLUS notation
models: gemma-2-2b-it
dataset: folio
evaluation: zero-shot
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# benchmark experiment runtime
!pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.6 MB/s eta 0:00:00
time: 249 µs (started: 2026-04-23 09:30:11 +00:00)


In [3]:
# start preparing for QA pipeline
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 90.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
time: 28.5 s (started: 2026-04-23 09:30:11 +00:00)


In [4]:
import pandas as pd

folio_train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_kr_gold_train_sef.csv")
folio_test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_kr_gold_test_sef.csv")
folio_val_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_kr_gold_valid_sef.csv")
# merge splits into one dataframe
folio_df = pd.concat([folio_train_df, folio_test_df, folio_val_df], ignore_index=True, sort=False)
# check if merge worked
print("*** TRAIN SPLIT LEN:", len(folio_train_df))
print("*** TEST SPLIT LEN:", len(folio_test_df))
print("*** VALIDATION SPLIT LEN:", len(folio_val_df))
print("*** MERGED SPLIT LEN:", len(folio_df))

*** TRAIN SPLIT LEN: 800
*** TEST SPLIT LEN: 201
*** VALIDATION SPLIT LEN: 203
*** MERGED SPLIT LEN: 1204
time: 1.91 s (started: 2026-04-23 09:30:40 +00:00)


In [5]:
# evaluation metrics

import numpy as np
import re
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def predict_answer(model, tokenizer, obj, subject, ref_relation=None, source_knowledge=None):
  # define notation grammar
  grammar = r"""
    start: program
    program: [stat]+
    stat: proposition newline
    proposition: atomicproposition | complexproposition
    complexproposition: leftparen proposition rightparen | proposition plus proposition | proposition minus proposition
    atomicproposition: leftparen* (plus+ | minus+) term | leftparen* (plus* | minus*) term | term
    leftparen: "("
    rightparen: ")"
    plus: "+"
    minus: "-"
    term: T n | leftparen* T n | leftparen* (plus* | minus*) T n | leftparen* (plus* | minus*) T n rightparen* | leftparen* (plus* | minus*) T n rightparen* (plus* | minus*) | leftparen* (plus* | minus*) T n leftparen* | | leftparen* (plus+ | minus+) T n leftparen*
    T: LETTER
    n: NUMBER
    newline: /\n/

    %import common.LETTER
    %import common.INT -> NUMBER
    %import common.WS
    %ignore WS
"""
  # prepare prompt
  rag_prompt = f"""
  <start_of_turn>user
  You are an expert logician. You are given a syllogism in TFLPLUS with premises between <PREMISES></PREMISES> and conclusion between <CONCLUSION></CONCLUSION> tags.
  The TFLPLUS BNF grammar to understand and reason in the language is given in the <GRAMMAR></GRAMMAR> tags.
  <GRAMMAR>{grammar}</GRAMMAR>
  <PREMISES>{subject}</PREMISES>
  <CONCLUSION>{obj}</CONCLUSION>
  Classify the conclusion as "True" if true, "False" if false or "Uncertain" if uncertain based on the premises. Present your answer only between <output></output> tags.
  <end_of_turn>
  <start_of_turn>model
  """
  input_ids = tokenizer(rag_prompt, return_tensors="pt").to(model.device)
  response = model.generate(**input_ids, max_new_tokens=500)
  predicted_relation = tokenizer.decode(response[0])
  matches = re.findall('<output>(.*)</output>', predicted_relation, flags=re.DOTALL)
  res = re.findall(r"<output>(.*)", matches[-1])  # from ['</output> tags.\n  <end_of_turn>\n  <start_of_turn>model\n  <output>T'] to ['T']
  predicted_label = res[0] if res else "None" # take first element from list ['T'] to get 'T'

  print("*** Premises: \n", subject)
  print("*** Conclusion: \n", obj)
  print("*** True Label: \n", ref_relation)
  print("*** Predicted Label: \n", predicted_label)
  return predicted_label

time: 1.04 s (started: 2026-04-23 09:30:41 +00:00)


In [6]:
def infer_from_ontology(dataset, model, tokenizer, mode='default', notation='NL'):
  evaluation_metrics_df = pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1"])
  reference_labels = []
  predicted_labels = []
  for index, row in dataset.iterrows():
      if notation == "NL":
        conclusion = row["conclusion"]
        premises = row["premises"]
      else:
        conclusion = row["conclusion-" + notation]
        premises = row["premises-" + notation]
      label = row["label"]
      if mode.lower() == "grammar":
        # conduct query with RAG retrival of sources
        # set number of candidate answers to consider as half the total triple store axioms
        source_information = """BNF GRAMMAR"""
        print("*** RAG INFORMATION:", source_information)
      # predict answer with model
      predicted_label = predict_answer(model, tokenizer, conclusion, premises, label)
      reference_labels.append(label)
      predicted_labels.append(predicted_label)
  # fill evaluation metrics dataframe
  accuracy_metric = accuracy_score(reference_labels, predicted_labels)
  precision_metric = precision_score(reference_labels, predicted_labels, average="macro")
  recall_metric = recall_score(reference_labels, predicted_labels, average="macro")
  f1_metric = f1_score(reference_labels, predicted_labels, average="macro")
  evaluation_metrics_df["Accuracy"] = [accuracy_metric]
  evaluation_metrics_df["Precision"] = [precision_metric]
  evaluation_metrics_df["Recall"] = [recall_metric]
  evaluation_metrics_df["F1"] = [f1_metric]
  print("Classification Report:", classification_report(reference_labels, predicted_labels))
  print("*************** INFERENCE COMPLETE ***************")
  return reference_labels, predicted_labels, evaluation_metrics_df, accuracy_metric, precision_metric, recall_metric, f1_metric

time: 1.82 ms (started: 2026-04-23 09:30:43 +00:00)


In [7]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings("ignore")

time: 20.4 s (started: 2026-04-23 09:30:43 +00:00)


In [8]:
# login to hugging face to have access to the model
!pip install huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

time: 4.42 s (started: 2026-04-23 09:31:03 +00:00)


In [10]:
# try rag search with gemma
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it", device_map="auto")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

time: 20.8 s (started: 2026-04-23 09:34:00 +00:00)


In [11]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(folio_df, model, tokenizer, mode='default', notation='TFLPLUS')

Streaming output truncated to the last 5000 lines.
-(+H2+(+(+H1++T1++H1++P1))

*** Conclusion: 
 +E2(+t2)+(+(+H1++T1++H1++P1))

*** True Label: 
 Uncertain
*** Predicted Label: 
 True
*** Premises: 
 -((+G0++R0)-+C0)
-(+G0++I0-+R0)
-((+G0++C0)-+h0)
-((+G0++H0)--+O0)
+G2
-(+H2(+c2)++I2)-+R2(+c2)

*** Conclusion: 
 +C2-+O2

*** True Label: 
 True
*** Predicted Label: 
 True
*** Premises: 
 -((+F0++S0)-+S0)
++(+F1++S1++S1+(-(+x1))++F1++S1++S1)
-((+F0++S0)--(+S0))
+F2(+a2)+(+S2-+S2)
-(+S2-+S2)

*** Conclusion: 
 +S2-+S2

*** True Label: 
 True
*** Predicted Label: 
 True
*** Premises: 
 -(+U0++U0-+S0)
-(+U0++M0-+U0)
-(+U0-+M0-+L0)
-(+U0++L0-+C0)
-(+U0++S0-+W0)
-(+U0++C0-+P0)
+U2++W2(+b2)++M2(+b2))
+U2+-(+L2(+p2)-+S2(+p2))

*** Conclusion: 
 -+P2(+b2)

*** True Label: 
 Uncertain
*** Predicted Label: 
 True
*** Premises: 
 +M2(+d2)++D2
+T2
+P2
-(+I2)

*** Conclusion: 
 -(+P2)

*** True Label: 
 True
*** Predicted Label: 
 True
*** Premises: 
 ++(+E1++W1++G1+(-(+x1))++E1++W1++G1)
-((+E0++W0)

In [12]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.38205980066445183
***** PRECISION *****
0.1273532668881506
***** RECALL *****
0.3333333333333333
***** F1 *****
0.1842948717948718


,Accuracy,Precision,Recall,F1
0,0.38206,0.127353,0.333333,0.184295


time: 22.7 ms (started: 2026-04-23 09:55:40 +00:00)


In [13]:
# empty torch cuda cache
torch.cuda.empty_cache()

# delete model from cpu
del(model)

time: 5.76 ms (started: 2026-04-23 09:55:40 +00:00)
